# Notebook 27 — Residual Spectral Transition Analysis

This notebook extends the residual universality manifold paper with a refined spectral-operator layer.

It generates Laplacian spectrum diagnostics for the residual manifold graph:

- residual Laplacian eigenspectrum with dominant transition markers,
- spectral density and cumulative residual spectrum,
- cumulative eigengap continuity hierarchy,
- dominant spectral gap table,
- spectral entropy,
- topology-family spectral trajectories,
- spectral overlap matrix,
- spectral hierarchy decomposition.

The notebook is designed to work even when `results/` is empty in Colab. If prior notebook outputs are unavailable, it regenerates a deterministic residual manifold baseline compatible with Notebooks 24–26.


In [ ]:
# Notebook 27 setup
from pathlib import Path
import json
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

try:
    import networkx as nx
except ImportError as e:
    raise ImportError("This notebook requires networkx. In Colab: !pip install networkx") from e

try:
    from scipy.linalg import eigh
except ImportError as e:
    raise ImportError("This notebook requires scipy. In Colab: !pip install scipy") from e

# Resolve repo-style paths robustly for Colab and local execution.
CWD = Path.cwd()
if CWD.name == "notebooks":
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
EXPORTS_DIR = REPO_ROOT / "exports"
for d in [RESULTS_DIR, FIGURES_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 9423
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

print("cwd:", CWD)
print("repo root:", REPO_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)
print("exports:", EXPORTS_DIR)


## 1. Load existing residual manifold data or regenerate baseline

The notebook first looks for residual manifold coordinates from earlier notebooks. If none are present, it regenerates a deterministic graph-family dataset using the same family names and graph sizes used in the paper.


In [ ]:
def read_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f"loaded: {p}")
            return pd.read_csv(p), p
    return None, None

candidate_files = [
    RESULTS_DIR / "residual_universality_embedding.csv",
    RESULTS_DIR / "residual_pca_embedding.csv",
    RESULTS_DIR / "residual_classification_feature_matrix.csv",
    RESULTS_DIR / "residual_geometry_features.csv",
    RESULTS_DIR / "25_transport_nodes.csv",
    RESULTS_DIR / "26_diffusion_nodes.csv",
]

raw, source_path = read_first_existing(candidate_files)
print("available result files:", sorted(p.name for p in RESULTS_DIR.glob("*"))[:40])


In [ ]:
FAMILIES = ["ring lattice", "small world", "Erdős–Rényi", "scale free", "modular clustered"]
SIZES = [16, 32, 64, 128]

def stable_seed(*parts):
    text = "|".join(map(str, parts))
    return (sum((i + 1) * ord(ch) for i, ch in enumerate(text)) + SEED) % (2**32 - 1)

def safe_graph_features(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    deg = np.array([d for _, d in G.degree()], dtype=float)
    if n == 0:
        raise ValueError("empty graph")
    density = nx.density(G)
    clustering = nx.average_clustering(G) if n > 1 else 0.0
    transitivity = nx.transitivity(G) if n > 2 else 0.0
    components = nx.number_connected_components(G)
    largest_cc = max(nx.connected_components(G), key=len)
    H = G.subgraph(largest_cc).copy()
    try:
        avg_path = nx.average_shortest_path_length(H) if H.number_of_nodes() > 1 else 0.0
    except Exception:
        avg_path = 0.0
    try:
        diameter = nx.diameter(H) if H.number_of_nodes() > 1 else 0.0
    except Exception:
        diameter = 0.0
    try:
        assort = nx.degree_assortativity_coefficient(G)
        if not np.isfinite(assort):
            assort = 0.0
    except Exception:
        assort = 0.0
    triangles = sum(nx.triangles(G).values()) / 3.0
    return {
        "n": n,
        "edges": m,
        "density": density,
        "mean_degree": float(deg.mean()),
        "std_degree": float(deg.std()),
        "max_degree": float(deg.max()),
        "degree_heterogeneity": float(deg.std() / (deg.mean() + 1e-9)),
        "clustering": clustering,
        "transitivity": transitivity,
        "components": components,
        "largest_component_frac": len(largest_cc) / n,
        "avg_path_largest_cc": avg_path,
        "diameter_largest_cc": diameter,
        "assortativity": assort,
        "triangles": triangles,
    }

def make_graph(family, n, seed):
    if family == "ring lattice":
        k = min(4, n - 1)
        if k % 2 == 1:
            k -= 1
        return nx.watts_strogatz_graph(n, k=max(k, 2), p=0.0, seed=seed)
    if family == "small world":
        k = min(4, n - 1)
        if k % 2 == 1:
            k -= 1
        return nx.watts_strogatz_graph(n, k=max(k, 2), p=0.18, seed=seed)
    if family == "Erdős–Rényi":
        p = min(0.18, 4 / max(n - 1, 1))
        return nx.erdos_renyi_graph(n, p=p, seed=seed)
    if family == "scale free":
        m = max(1, min(3, n // 10 + 1))
        return nx.barabasi_albert_graph(n, m=m, seed=seed)
    if family == "modular clustered":
        sizes = [n // 2, n - n // 2]
        p_in = min(0.55, 6 / max(sizes[0], 2))
        p_out = 0.025
        return nx.stochastic_block_model(sizes, [[p_in, p_out], [p_out, p_in]], seed=seed)
    raise ValueError(f"unknown family: {family}")

def regenerate_baseline():
    rows = []
    graphs = {}
    for family in FAMILIES:
        for n in SIZES:
            seed = stable_seed(family, n)
            G = make_graph(family, n, seed)
            graphs[(family, n)] = G
            row = safe_graph_features(G)
            row.update({"family": family, "topology": family, "graph_size": n, "N": n})
            rows.append(row)
    df = pd.DataFrame(rows)
    feature_cols = [c for c in df.columns if c not in ["family", "topology", "graph_size", "N"]]
    X = StandardScaler().fit_transform(df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0))
    pca = PCA(n_components=2, random_state=SEED)
    coords = pca.fit_transform(X)
    df["PC1"] = coords[:, 0]
    df["PC2"] = coords[:, 1]
    df["source"] = "regenerated_baseline_notebook_27"
    return df, graphs, feature_cols, pca.explained_variance_ratio_

if raw is None:
    print("No compatible residual manifold file found; regenerating deterministic baseline.")
    manifold_df, graph_store, feature_cols, pca_var = regenerate_baseline()
    source_path = RESULTS_DIR / "27_regenerated_residual_manifold.csv"
    manifold_df.to_csv(source_path, index=False)
else:
    manifold_df = raw.copy()
    rename_map = {}
    for c in manifold_df.columns:
        lc = c.lower()
        if lc in ["topology", "label", "graph_family"]:
            rename_map[c] = "family"
        if lc in ["n", "size"]:
            rename_map[c] = "graph_size"
        if lc in ["x", "pca1", "coord1"]:
            rename_map[c] = "PC1"
        if lc in ["y", "pca2", "coord2"]:
            rename_map[c] = "PC2"
    manifold_df = manifold_df.rename(columns=rename_map)
    if "family" not in manifold_df.columns and "topology" in manifold_df.columns:
        manifold_df["family"] = manifold_df["topology"]
    if "graph_size" not in manifold_df.columns and "N" in manifold_df.columns:
        manifold_df["graph_size"] = manifold_df["N"]
    if "N" not in manifold_df.columns and "graph_size" in manifold_df.columns:
        manifold_df["N"] = manifold_df["graph_size"]

    if not {"PC1", "PC2"}.issubset(manifold_df.columns):
        numeric_cols = manifold_df.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c not in ["N", "graph_size"]]
        X = StandardScaler().fit_transform(manifold_df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0))
        coords2 = PCA(n_components=2, random_state=SEED).fit_transform(X)
        manifold_df["PC1"] = coords2[:, 0]
        manifold_df["PC2"] = coords2[:, 1]

    graph_store = {}
    for _, row in manifold_df.dropna(subset=["family", "graph_size"]).iterrows():
        family = str(row["family"])
        n = int(row["graph_size"])
        if family in FAMILIES and n in SIZES:
            graph_store[(family, n)] = make_graph(family, n, stable_seed(family, n))
    feature_cols = []
    pca_var = None

manifold_df["family"] = manifold_df["family"].astype(str)
manifold_df["graph_size"] = manifold_df["graph_size"].astype(int)
manifold_df["N"] = manifold_df["graph_size"]

print("source_path:", source_path)
print(manifold_df[["family", "graph_size", "PC1", "PC2"]].head())
print("rows:", len(manifold_df))


## 2. Build residual manifold kNN graph and Laplacian spectrum

We construct a kNN graph over residual manifold coordinates and analyze the graph Laplacian:

\[
L = D - A
\]

with eigensystem:

\[
Lu_k = \lambda_k u_k.
\]


In [ ]:
coords = manifold_df[["PC1", "PC2"]].to_numpy(dtype=float)
n_nodes = len(coords)
k = min(4, max(2, n_nodes - 1))

nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
distances, indices = nbrs.kneighbors(coords)
sigma = np.median(distances[:, 1:]) + 1e-9

A = np.zeros((n_nodes, n_nodes), dtype=float)
for i in range(n_nodes):
    for dist, j in zip(distances[i, 1:], indices[i, 1:]):
        w = np.exp(-(dist**2) / (2 * sigma**2))
        A[i, j] = max(A[i, j], w)
        A[j, i] = max(A[j, i], w)

D = np.diag(A.sum(axis=1))
L = D - A
L = 0.5 * (L + L.T)
evals, evecs = eigh(L)
evals = np.maximum(evals, 0.0)

spectrum_df = pd.DataFrame({
    "mode": np.arange(len(evals)),
    "eigenvalue": evals,
})
spectrum_df["eigengap"] = spectrum_df["eigenvalue"].diff().fillna(0.0)
spectrum_df.to_csv(RESULTS_DIR / "27_laplacian_spectrum.csv", index=False)

mode_df = manifold_df[["family", "graph_size", "PC1", "PC2"]].copy()
for idx, mode in enumerate(range(1, min(5, evecs.shape[1]))):
    mode_df[f"u{mode}"] = evecs[:, mode]
mode_df.to_csv(RESULTS_DIR / "27_laplacian_modes_by_node.csv", index=False)

print(spectrum_df.head(10))


## 3. Figure 1 — Residual Laplacian eigenspectrum with hierarchy bands

This figure identifies low-frequency continuity, intermediate transport coupling, and high-frequency fragmentation regimes directly on the residual eigenspectrum.


In [ ]:
def top_gap_table(evals, n_top=3, skip=1):
    spacing = np.diff(evals)
    rows = []
    for k_idx, gap in enumerate(spacing):
        rows.append({
            "mode_k": k_idx,
            "mode_k_plus_1": k_idx + 1,
            "lambda_k": evals[k_idx],
            "lambda_k_plus_1": evals[k_idx + 1],
            "gap": gap,
        })
    df = pd.DataFrame(rows)
    if len(df) > skip:
        ranked = df.iloc[skip:].copy().sort_values("gap", ascending=False).head(n_top)
    else:
        ranked = df.copy().sort_values("gap", ascending=False).head(n_top)
    ranked["normalized_gap"] = ranked["gap"] / (df["gap"].max() + 1e-12)
    return df, ranked

spacing_full_df, dominant_gaps = top_gap_table(evals, n_top=3, skip=1)
spacing_full_df.to_csv(RESULTS_DIR / "27_eigenvalue_spacing.csv", index=False)
dominant_gaps.to_csv(RESULTS_DIR / "27_dominant_spectral_gaps.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.plot(spectrum_df["mode"], spectrum_df["eigenvalue"], marker="o", linewidth=2.5)

n_modes = len(evals)
low_end = max(1, n_modes // 3)
mid_end = max(low_end + 1, 2 * n_modes // 3)
ax.axvspan(0, low_end, alpha=0.08, label="low-frequency continuity")
ax.axvspan(low_end, mid_end, alpha=0.08, label="transport-coupling band")
ax.axvspan(mid_end, n_modes - 1, alpha=0.08, label="fragmentation band")

for _, r in dominant_gaps.iterrows():
    x = r["mode_k_plus_1"]
    y = r["lambda_k_plus_1"]
    ax.axvline(x, linestyle="--", alpha=0.45)
    ax.text(x + 0.05, y, f"gap {r['gap']:.2f}", fontsize=9, ha="left", va="bottom")

ax.set_title("Residual Laplacian eigenspectrum")
ax.set_xlabel("mode")
ax.set_ylabel("eigenvalue")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", frameon=True)
fig.tight_layout()
out = FIGURES_DIR / "27_residual_laplacian_eigenspectrum.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out)
print(dominant_gaps)


## 4. Figures 2a–2b — Spectral density and cumulative residual spectrum

The spectral density summarizes eigenvalue concentration. The cumulative spectrum shows how spectral mass accumulates across residual manifold modes.


In [ ]:
# Spectral density as a standalone figure.
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(evals, bins=min(12, max(5, len(evals)//2)), alpha=0.85)
ax.set_title("Residual spectral density")
ax.set_xlabel("eigenvalue")
ax.set_ylabel("count")
ax.grid(True, alpha=0.25)
fig.tight_layout()
out_density = FIGURES_DIR / "27_residual_spectral_density.png"
fig.savefig(out_density, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out_density)

# Cumulative spectrum as a standalone figure.
cum = np.cumsum(evals) / (np.sum(evals) + 1e-12)
cum_df = pd.DataFrame({"mode": np.arange(len(cum)), "cumulative_spectral_mass": cum})
cum_df.to_csv(RESULTS_DIR / "27_cumulative_residual_spectrum.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(np.arange(len(cum)), cum, marker="o", linewidth=2.5)
for level in [0.33, 0.66, 0.90]:
    idx = int(np.searchsorted(cum, level))
    idx = min(idx, len(cum) - 1)
    ax.axhline(level, linestyle="--", alpha=0.25)
    ax.axvline(idx, linestyle="--", alpha=0.25)
    ax.text(idx + 0.1, level, f"{level:.0%} mass", fontsize=9, va="bottom")
ax.set_title("Cumulative residual spectrum")
ax.set_xlabel("mode")
ax.set_ylabel("cumulative spectral mass")
ax.set_ylim(-0.02, 1.03)
ax.grid(True, alpha=0.25)
fig.tight_layout()
out_cum = FIGURES_DIR / "27_cumulative_residual_spectrum.png"
fig.savefig(out_cum, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out_cum)

# Keep backward-compatible combined figure for paper variants already referencing it.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(evals, bins=min(12, max(5, len(evals)//2)), alpha=0.85)
axes[0].set_title("Residual spectral density")
axes[0].set_xlabel("eigenvalue")
axes[0].set_ylabel("count")
axes[0].grid(True, alpha=0.25)
axes[1].plot(np.arange(len(cum)), cum, marker="o", linewidth=2.2)
axes[1].set_title("Cumulative residual spectrum")
axes[1].set_xlabel("mode")
axes[1].set_ylabel("cumulative spectral mass")
axes[1].grid(True, alpha=0.25)
fig.tight_layout()
out_combo = FIGURES_DIR / "27_spectral_density_and_cumulative.png"
fig.savefig(out_combo, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out_combo)


## 5. Figure 3 — Cumulative eigengap continuity hierarchy

This replaces the weaker eigengap histogram with a hierarchy-oriented curve.

\[
\Delta\lambda_k = \lambda_{k+1}-\lambda_k
\]

The cumulative gap curve emphasizes how spectral transition mass accumulates from low-frequency continuity bands toward higher-frequency residual fragmentation.


In [ ]:
spacing = np.diff(evals)
spacing_modes = np.arange(len(spacing))
spacing_sum = spacing.sum() + 1e-12
cum_spacing = np.cumsum(spacing) / spacing_sum

hierarchy_df = pd.DataFrame({
    "mode_k": spacing_modes,
    "eigengap": spacing,
    "cumulative_eigengap_mass": cum_spacing,
})
hierarchy_df.to_csv(RESULTS_DIR / "27_cumulative_eigengap_hierarchy.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(spacing_modes, cum_spacing, marker="o", linewidth=2.5)

# Hierarchy bands in cumulative gap mass.
ax.axhspan(0.00, 0.33, alpha=0.08)
ax.axhspan(0.33, 0.66, alpha=0.12)
ax.axhspan(0.66, 1.00, alpha=0.08)
ax.text(0.3, 0.14, "continuity regime", fontsize=10)
ax.text(0.3, 0.48, "transport regime", fontsize=10)
ax.text(0.3, 0.82, "fragmentation regime", fontsize=10)

for _, r in dominant_gaps.iterrows():
    idx = int(r["mode_k"])
    ax.axvline(idx, linestyle="--", alpha=0.45)
    ax.scatter([idx], [cum_spacing[idx]], s=80, zorder=5)
    ax.text(
        idx + 0.08,
        cum_spacing[idx],
        f"gap {r['gap']:.2f}",
        fontsize=9,
        ha="left",
        va="bottom",
    )

ax.set_title("Cumulative eigengap continuity hierarchy")
ax.set_xlabel("mode k")
ax.set_ylabel("normalized cumulative eigengap mass")
ax.set_ylim(-0.02, 1.03)
ax.grid(True, alpha=0.25)
fig.tight_layout()
out = FIGURES_DIR / "27_cumulative_eigengap_hierarchy.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out)

# Backward-compatible filename for existing paper Figure 8 references.
compat = FIGURES_DIR / "27_eigenvalue_spacing.png"
fig.savefig(compat, dpi=220, bbox_inches="tight")
print("also wrote replacement:", compat)


## 6. Figure 4 — Spectral hierarchy decomposition

This compact summary partitions residual spectral mass into three operator bands:

- low-frequency continuity,
- intermediate transport coupling,
- high-frequency fragmentation.


In [ ]:
bands = [
    ("continuity", 0, low_end),
    ("transport", low_end, mid_end),
    ("fragmentation", mid_end, len(evals)),
]
rows = []
for name, start, stop in bands:
    vals = evals[start:stop]
    rows.append({
        "band": name,
        "start_mode": start,
        "stop_mode_exclusive": stop,
        "spectral_mass": float(vals.sum()),
        "normalized_mass": float(vals.sum() / (evals.sum() + 1e-12)),
        "mean_eigenvalue": float(vals.mean()) if len(vals) else 0.0,
    })
band_df = pd.DataFrame(rows)
band_df.to_csv(RESULTS_DIR / "27_spectral_hierarchy_decomposition.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(band_df["band"], band_df["normalized_mass"])
for i, r in band_df.iterrows():
    ax.text(i, r["normalized_mass"] + 0.01, f"{r['normalized_mass']:.2f}", ha="center", va="bottom")
ax.set_ylim(0, max(1.0, band_df["normalized_mass"].max() * 1.2))
ax.set_title("Residual spectral hierarchy decomposition")
ax.set_ylabel("normalized spectral mass")
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
out = FIGURES_DIR / "27_spectral_hierarchy_decomposition.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out)
band_df


## 7. Figure 5 — Spectral entropy curve

Spectral entropy measures how Laplacian energy is distributed across modes.

\[
H_\lambda = -\sum_k p_k \log p_k,
\qquad
p_k = \frac{\lambda_k}{\sum_j \lambda_j}
\]


In [ ]:
positive = evals[evals > 1e-12]
p = positive / (positive.sum() + 1e-12)
spectral_entropy = float(-np.sum(p * np.log(p + 1e-12)))

entropy_rows = []
for m in range(1, len(evals) + 1):
    vals = evals[:m]
    vals = vals[vals > 1e-12]
    if len(vals) == 0:
        H = 0.0
    else:
        pm = vals / (vals.sum() + 1e-12)
        H = float(-np.sum(pm * np.log(pm + 1e-12)))
    entropy_rows.append({"mode_cutoff": m - 1, "spectral_entropy": H})

entropy_df = pd.DataFrame(entropy_rows)
entropy_df.to_csv(RESULTS_DIR / "27_spectral_entropy_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(entropy_df["mode_cutoff"], entropy_df["spectral_entropy"], marker="o", linewidth=2.5)
ax.axhline(spectral_entropy, linestyle="--", alpha=0.5, label=f"full entropy = {spectral_entropy:.2f}")
ax.set_title("Residual spectral entropy curve")
ax.set_xlabel("mode cutoff")
ax.set_ylabel("spectral entropy")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / "27_spectral_entropy_curve.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out)


## 8. Figure 6 — Topology-family spectral trajectories

For each generated graph family and size, compute a compact graph-level spectrum. These trajectories show how spectral summaries extend across graph-size scaling.


In [ ]:
def graph_spectral_summary(G):
    A_g = nx.to_numpy_array(G, dtype=float)
    D_g = np.diag(A_g.sum(axis=1))
    L_g = 0.5 * ((D_g - A_g) + (D_g - A_g).T)
    vals = np.maximum(eigh(L_g, eigvals_only=True), 0.0)
    vals = np.sort(vals)
    pos = vals[vals > 1e-10]
    if len(pos) == 0:
        pos = np.array([0.0])
    gaps = np.diff(vals)
    pp = pos / (pos.sum() + 1e-12)
    return {
        "lambda_1": float(vals[1]) if len(vals) > 1 else 0.0,
        "lambda_2": float(vals[2]) if len(vals) > 2 else 0.0,
        "lambda_mean": float(vals.mean()),
        "lambda_max": float(vals.max()),
        "spectral_gap_max": float(gaps.max()) if len(gaps) else 0.0,
        "spectral_energy": float(np.sum(vals**2)),
        "spectral_entropy": float(-np.sum(pp * np.log(pp + 1e-12))),
    }

spec_rows = []
for (family, n), G in sorted(graph_store.items(), key=lambda x: (str(x[0][0]), x[0][1])):
    row = graph_spectral_summary(G)
    row.update({"family": family, "graph_size": n})
    spec_rows.append(row)

family_spec = pd.DataFrame(spec_rows)
family_spec.to_csv(RESULTS_DIR / "27_family_graph_spectral_summary.csv", index=False)

num_cols = ["lambda_1", "lambda_2", "lambda_mean", "lambda_max", "spectral_gap_max", "spectral_energy", "spectral_entropy"]
X_spec = StandardScaler().fit_transform(family_spec[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0))
spec_coords = PCA(n_components=2, random_state=SEED).fit_transform(X_spec)
family_spec["spectral_PC1"] = spec_coords[:, 0]
family_spec["spectral_PC2"] = spec_coords[:, 1]
family_spec.to_csv(RESULTS_DIR / "27_family_spectral_trajectories.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 8))
for family, g in family_spec.groupby("family"):
    g = g.sort_values("graph_size")
    ax.plot(g["spectral_PC1"], g["spectral_PC2"], marker="o", linewidth=2.3, label=family)
    for _, r in g.iterrows():
        ax.text(r["spectral_PC1"], r["spectral_PC2"], f"N={int(r['graph_size'])}", fontsize=8)
ax.axhline(0, linestyle="--", alpha=0.5)
ax.axvline(0, linestyle="--", alpha=0.5)
ax.set_title("Topology-family spectral trajectories")
ax.set_xlabel("spectral PC1")
ax.set_ylabel("spectral PC2")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / "27_topology_family_spectral_trajectories.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out)


## 9. Figure 7 — Spectral overlap matrix

Family spectra are compared using cosine similarity over graph-level spectral descriptors. This defines a direct spectral overlap measurement between topology families.


In [ ]:
family_centroids = family_spec.groupby("family")[num_cols].mean()
Xc = StandardScaler().fit_transform(family_centroids.replace([np.inf, -np.inf], np.nan).fillna(0.0))
sim = cosine_similarity(Xc)
sim01 = (sim - sim.min()) / (sim.max() - sim.min() + 1e-12)

sim_df = pd.DataFrame(sim01, index=family_centroids.index, columns=family_centroids.index)
sim_df.to_csv(RESULTS_DIR / "27_spectral_overlap_matrix.csv")

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim01, vmin=0, vmax=1)
ax.set_title("Topology-family spectral overlap matrix")
ax.set_xticks(np.arange(len(sim_df.columns)))
ax.set_xticklabels(sim_df.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(sim_df.index)))
ax.set_yticklabels(sim_df.index)
for i in range(sim01.shape[0]):
    for j in range(sim01.shape[1]):
        ax.text(j, i, f"{sim01[i, j]:.2f}", ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, label="normalized spectral overlap")
fig.tight_layout()
out = FIGURES_DIR / "27_spectral_overlap_matrix.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.show()
print("wrote:", out)


## 10. Paper-ready spectral summary

This cell writes a short markdown summary that can be adapted into the manuscript.


In [ ]:
summary_md = f'''# Notebook 27 spectral transition summary

Source data: `{source_path}`

## Core spectral quantities

- Residual manifold nodes: {n_nodes}
- kNN graph size: k={k}
- Full spectral entropy: {spectral_entropy:.4f}
- Smallest nontrivial residual Laplacian eigenvalue: {evals[1] if len(evals)>1 else 0.0:.4f}
- Largest residual Laplacian eigenvalue: {evals[-1]:.4f}

## Dominant eigengaps

{dominant_gaps.to_markdown(index=False)}

## Paper interpretation

The residual Laplacian spectrum provides a true operator-level diagnostic for the residual universality manifold.
Low-frequency modes encode large-scale topology-family separability, while cumulative eigengap hierarchy describes transitions from low-frequency continuity structure to higher-frequency residual fragmentation.
Family-level spectral trajectories extend this structure across graph-size scaling, and spectral overlap measurements quantify topology-family proximity in spectral coordinates.

## Generated paper figures

- `27_residual_laplacian_eigenspectrum.png`
- `27_residual_spectral_density.png`
- `27_cumulative_residual_spectrum.png`
- `27_spectral_density_and_cumulative.png`
- `27_cumulative_eigengap_hierarchy.png`
- `27_eigenvalue_spacing.png`  # backward-compatible replacement
- `27_spectral_hierarchy_decomposition.png`
- `27_spectral_entropy_curve.png`
- `27_topology_family_spectral_trajectories.png`
- `27_spectral_overlap_matrix.png`
'''

summary_path = EXPORTS_DIR / "27_spectral_transition_summary.md"
summary_path.write_text(summary_md)
print(summary_md)
print("wrote:", summary_path)


## 11. Export bundle

This creates a zip containing all Notebook 27 figures, CSV outputs, and a markdown summary.


In [ ]:
manifest = {
    "notebook": "27_residual_spectral_transition_analysis.ipynb",
    "source_file": str(source_path),
    "created_outputs": {
        "figures": sorted([p.name for p in FIGURES_DIR.glob("27_*.png")]),
        "results": sorted([p.name for p in RESULTS_DIR.glob("27_*")]),
        "exports": sorted([p.name for p in EXPORTS_DIR.glob("27_*")]),
    },
}

manifest_path = EXPORTS_DIR / "27_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / "27_residual_spectral_transition_analysis_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in FIGURES_DIR.glob("27_*.png"):
        z.write(p, arcname=f"figures/{p.name}")
    for p in RESULTS_DIR.glob("27_*"):
        z.write(p, arcname=f"results/{p.name}")
    for p in EXPORTS_DIR.glob("27_*"):
        if p != zip_path:
            z.write(p, arcname=f"exports/{p.name}")

print("Wrote:", zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / 1e6, 3))

# Optional Colab download.
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Colab download skipped. Download manually from:", zip_path)
